# Task 2 — Step 3: Reward-Overoptimization and KL Pressure ($\beta_{\text{KL}}$)

### Research Question
When KL penalty pressure $\beta_{\text{KL}} \in \{0, 0.10, 0.20\}$ is modified:
1. Which observable metric shifts first: learned reward, policy entropy, reference drift (KL), or response length?
2. Does weakening KL pressure ($eta_{\text{KL}} = 0$) produce **reward overoptimization (Goodhart's Law)**, where reward increases without genuine quality improvement?


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Run KL Pressure Sweep or Load Results


In [ ]:
FORCE_RUN = False
if FORCE_RUN:
    !python -m task2_ppo.ablate_kl --config configs/ppo.yaml
else:
    print("[OK] Loading saved KL pressure results...")


## 2. Summary Table across $\beta_{\text{KL}}$ Values


In [ ]:
kl_file = "results/task2_ppo/ppo_kl_ablation_results.json"
if os.path.exists(kl_file):
    with open(kl_file, encoding='utf-8') as f:
        kl_data = json.load(f)
        
    rows = []
    for b_str in sorted(kl_data.keys(), key=float):
        ev = kl_data[b_str].get("eval", {})
        rows.append({
            "Beta_KL": float(b_str),
            "Held-Out Reward": f"{ev.get('mean_reward', 0):.4f}",
            "Policy Drift (KL)": f"{ev.get('mean_kl', 0):.4f}",
            "Mean Length (tokens)": f"{ev.get('mean_response_length_tokens', 0):.1f}",
            "EOS Termination Rate": f"{ev.get('eos_termination_rate', 0)*100:.1f}%"
        })
    df_kl = pd.DataFrame(rows).sort_values("Beta_KL")
    display(df_kl)


## 3. Visualizing the Reward vs. Drift Trade-Off under KL Shaping


In [ ]:
if os.path.exists(kl_file):
    b_vals = df_kl["Beta_KL"].values
    rews = [float(r["Held-Out Reward"]) for _, r in df_kl.iterrows()]
    kl_vals = [float(r["Policy Drift (KL)"]) for _, r in df_kl.iterrows()]
    lengths = [float(r["Mean Length (tokens)"]) for _, r in df_kl.iterrows()]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Reward & KL vs Beta_KL
    ax1 = axes[0]
    ax1.plot(b_vals, rews, 'o-', color='#10B981', lw=2, label='Mean Reward')
    ax1.set_xlabel('β_KL')
    ax1.set_ylabel('Mean Reward', color='#10B981')
    ax1.tick_params(axis='y', labelcolor='#10B981')
    ax1.grid(True, alpha=0.3)
    
    ax2 = ax1.twinx()
    ax2.plot(b_vals, kl_vals, 's--', color='#EF4444', lw=2, label='Mean KL')
    ax2.set_ylabel('Mean KL Drift', color='#EF4444')
    ax2.tick_params(axis='y', labelcolor='#EF4444')
    axes[0].set_title('Reward and Reference Drift vs. β_KL')
    
    # 2. Length vs Beta_KL
    axes[1].bar([str(b) for b in b_vals], lengths, color='#6366F1', width=0.4, alpha=0.85)
    for i, l in enumerate(lengths):
        axes[1].text(i, l + 2, f"{l:.1f}", ha='center', fontweight='bold')
    axes[1].set_title('Mean Response Length vs. β_KL')
    axes[1].set_xlabel('β_KL')
    axes[1].set_ylabel('Length (tokens)')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()


## 4. Qualitative Disagreement Evidence
Identify responses where $\beta_{\text{KL}} = 0$ achieved very high reward scores but exhibited degenerative patterns (repetitive phrasing, evasive filler, or truncation).
